# Concurrency & Recovery Demos

Maps Neo4j to Elmasri & Navathe ch. 21–22. Runs against the 10-patient subset from `load_subset.py`.

**Prerequisites**
- `docker compose up -d`
- `python load_subset.py`

**Demo isolation.** Writes only touch demo-scoped properties (`p.demo_*`) on real PATIENT nodes; ephemeral entities carry `:DemoNode`. The setup cell wipes prior demo state, so the notebook is idempotent.

Each cell prints a slice of `query.log` or `debug.log` at the end — no terminal-tailing needed.

| # | Theme       | Demo                                              |
|---|-------------|---------------------------------------------------|
| 1 | Transaction | Atomic multi-entity rollback (idempotency case)   |
| 2 | Transaction | Checkpoint + WAL truncation                       |
| 3 | Transaction | Crash recovery — WAL bytes on disk + replay       |
| 4 | Concurrency | Non-repeatable read                               |
| 5 | Concurrency | Lost update (naive vs atomic SET)                 |

In [1]:
import os, time, uuid, shlex, subprocess
from threading import Event, Thread, Barrier

import neo4j
from neo4j.exceptions import ConstraintError
from dotenv import load_dotenv

load_dotenv()
URI       = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
USER      = os.getenv("NEO4J_USERNAME", "neo4j")
PWD       = os.getenv("NEO4J_PASSWORD", "password123")
DB        = os.getenv("NEO4J_DATABASE", "neo4j")
CONTAINER = "neo4j-demo"
RUN_ID    = uuid.uuid4().hex[:8]


def session():
    return driver.session(database=DB)


def show_log(logfile, n=20, grep=None):
    cmd = (f"grep -iE '{grep}' /logs/{logfile} 2>/dev/null | tail -n {n}"
           if grep else f"tail -n {n} /logs/{logfile}")
    out = subprocess.run(
        ["docker", "exec", CONTAINER, "sh", "-c", cmd],
        capture_output=True, text=True, timeout=15,
    ).stdout or "(no matches)"
    print(f"--- /logs/{logfile}  ({grep or 'tail'}) ---\n{out}")


def show_log_since(logfile, start_line, grep=None, max_line=240):
    """Show only the lines appended after start_line (optionally grep-filtered)."""
    cmd = f"tail -n +{start_line + 1} /logs/{logfile}"
    if grep:
        cmd += f" | grep -iE {shlex.quote(grep)}"
    out = subprocess.run(
        ["docker", "exec", CONTAINER, "sh", "-c", cmd],
        capture_output=True, text=True, timeout=15,
    ).stdout or "(no new lines)"
    if max_line:
        out = "\n".join((ln if len(ln) <= max_line else ln[:max_line] + " …[truncated]")
                        for ln in out.splitlines())
    print(f"--- /logs/{logfile} since line {start_line}  ({grep or 'all'}) ---\n{out}")


def log_lines(logfile):
    p = subprocess.run(
        ["docker", "exec", CONTAINER, "sh", "-c", f"wc -l < /logs/{logfile}"],
        capture_output=True, text=True,
    )
    return int(p.stdout.strip() or 0)


def cleanup():
    with session() as s:
        s.run("MATCH (n:DemoNode) DETACH DELETE n").consume()
        s.run("MATCH (p:PATIENT) "
              "REMOVE p.demo_note, p.demo_last_visit, "
              "       p.demo_visit_count, p.demo_run_id").consume()


driver = neo4j.GraphDatabase.driver(URI, auth=(USER, PWD))
driver.verify_connectivity()
cleanup()

with session() as s:
    row = s.run("""
        MATCH (p:PATIENT)-[:HAS_IMAGE]->(i:IMAGE)
        WHERE NOT p:DemoNode
        RETURN p.patient_id AS pid, i.instance_uid AS uid LIMIT 1
    """).single()
if not row:
    raise RuntimeError("No PATIENT/IMAGE found. Run `python load_subset.py`.")

PID          = row["pid"]
EXISTING_UID = row["uid"]
print(f"RUN_ID={RUN_ID}  PID={PID}")
print(f"sample instance_uid: {EXISTING_UID}")

RUN_ID=a20135ef  PID=27
sample instance_uid: 1.3.6.1.4.1.9590.100.1.2.59620512812470186337816449881316634272


## Demo 1 — Atomicity: rollback on retry

`workflow(tx, new_visit)` is called **twice with the same arguments**. Each call does 4 writes in one tx:

1. `MATCH` the patient.
2. `SET p.demo_last_visit`.
3. `CREATE :Annotation:DemoNode` + `[:ANNOTATES]->(p)`.
4. `CREATE :PATIENT:DemoNode {patient_id: NEW_PID}`.

- **Run 1.** `NEW_PID` is new → all 4 writes commit.
- **Run 2.** Same call. `NEW_PID` now duplicates → `ConstraintError` → auto-rollback → none of Run 2's writes survive.

Assertion `state() after Run 2 == state() after Run 1` proves the failed retry left no half-applied mess — the textbook reason non-idempotent writes need atomic transactions.

In [2]:
NEW_PID = 9_900_001  # demo-only patient id, not in the real subset


def state():
    with session() as s:
        r = s.run("""
            MATCH (p:PATIENT {patient_id: $pid})
            OPTIONAL MATCH (a:Annotation:DemoNode)
            OPTIONAL MATCH (:Annotation:DemoNode)-[rel:ANNOTATES]->(p)
            OPTIONAL MATCH (np:PATIENT:DemoNode)
            RETURN toString(p.demo_last_visit) AS visit,
                   count(DISTINCT a)   AS annot_nodes,
                   count(rel)          AS annot_rels,
                   count(DISTINCT np)  AS demo_patients
        """, pid=PID).single()
    return r["visit"], r["annot_nodes"], r["annot_rels"], r["demo_patients"]


def workflow(tx, new_visit):
    aid = uuid.uuid4().hex[:12]
    tx.run("""
        MATCH (p:PATIENT {patient_id: $pid})
        SET p.demo_last_visit = datetime($visit)
        CREATE (a:Annotation:DemoNode {annotation_id: $aid})
        MERGE (a)-[:ANNOTATES]->(p)
    """, pid=PID, visit=new_visit, aid=aid)
    tx.run("CREATE (:PATIENT:DemoNode {patient_id: $new_pid})", new_pid=NEW_PID)


# Reset
with session() as s:
    s.run("MATCH (n:DemoNode) DETACH DELETE n").consume()
    s.run("MATCH (p:PATIENT {patient_id: $pid}) "
          "SET p.demo_last_visit = datetime('2025-01-01')", pid=PID).consume()
initial = state()
print(f"INITIAL : {initial}   (visit, annot_nodes, annot_rels, demo_patients)")

# Run 1: NEW_PID doesn't exist yet -> all writes commit.
with session() as s, s.begin_transaction() as tx:
    workflow(tx, "2026-05-14")
after1 = state()
print(f"RUN 1   : {after1}   ← committed (first call: no conflict)")
assert after1 != initial

# Run 2: same call, NEW_PID now exists -> ConstraintError -> auto-rollback.
try:
    with session() as s, s.begin_transaction() as tx:
        workflow(tx, "2027-01-01")
except ConstraintError:
    print("\nConstraintError on duplicate PATIENT -> auto-rollback")
after2 = state()
print(f"RUN 2   : {after2}   ← rolled back (every Run-2 write reverted)")
assert after2 == after1, "Atomicity violated: some Run-2 write leaked through"

show_log("query.log", grep="Annotation|demo_last_visit")

INITIAL : ('2025-01-01T00:00:00Z', 0, 0, 0)   (visit, annot_nodes, annot_rels, demo_patients)
RUN 1   : ('2026-05-14T00:00:00Z', 1, 1, 1)   ← committed (first call: no conflict)

ConstraintError on duplicate PATIENT -> auto-rollback
RUN 2   : ('2026-05-14T00:00:00Z', 1, 1, 1)   ← rolled back (every Run-2 write reverted)
--- /logs/query.log  (Annotation|demo_last_visit) ---
(no matches)


## Demo 2 — Checkpoint

Every commit appends to the WAL. A *checkpoint* flushes dirty pages and lets older WAL be pruned — that's what bounds the recovery work in Demo 3.

Container is set (`docker-compose.yml`) to checkpoint every **1 log chunk** or every **1 second**, whichever fires first. The cell snapshots `debug.log`, runs 300 small writes, sleeps briefly, then counts NEW `"Checkpoint started"` lines.

*Note:* `db.checkpoint.interval.tx` counts log-appender chunks, not raw transactions — one chunk ≈ ~100–200 small commits in this workload (see the `txId` deltas between checkpoint lines).

In [3]:
# Diagnostic: which checkpoint thresholds is the live container using?
with session() as s:
    cfg = s.run(
        "CALL dbms.listConfig() YIELD name, value "
        "WHERE name CONTAINS 'checkpoint.interval' RETURN name, value"
    ).data()
print("Active checkpoint config:")
for r in cfg:
    print(f"  {r['name']:35} = {r['value']}")

# Snapshot debug.log size BEFORE our writes — used to filter "this run only".
start_line = log_lines("debug.log")
print(f"\ndebug.log starts at line {start_line}")

N = 300
t0 = time.perf_counter()
with session() as s:
    for i in range(N):
        s.run("MATCH (p:PATIENT {patient_id: $pid}) SET p.demo_note = $v",
              pid=PID, v=f"tick-{i}").consume()
elapsed = time.perf_counter() - t0
print(f"\n{N} writes done in {elapsed:.1f}s.")

time.sleep(3)  # catch the trailing time-based checkpoints

new_count = subprocess.run(
    ["docker", "exec", CONTAINER, "sh", "-c",
     f"tail -n +{start_line + 1} /logs/debug.log | grep -ci 'checkpoint started' || true"],
    capture_output=True, text=True,
).stdout.strip()

print(f"\ndebug.log grew by {log_lines('debug.log') - start_line} lines.")
print(f"NEW checkpoints during this cell: {new_count}\n")

show_log_since("debug.log", start_line, grep="checkpoint started|prun")

Active checkpoint config:
  db.checkpoint.interval.time         = 1s
  db.checkpoint.interval.tx           = 1
  db.checkpoint.interval.volume       = 250.00MiB

debug.log starts at line 7420

300 writes done in 2.2s.

debug.log grew by 9 lines.
NEW checkpoints during this cell: 3

--- /logs/debug.log since line 7420  (checkpoint started|prun) ---
2026-05-15 04:33:38.477+0000 INFO  [o.n.k.i.t.l.c.CheckPointerImpl] [neo4j/845e4053] Checkpoint triggered by "Scheduled checkpoint for every 1 log chunks threshold" @ txId: 6232, append index: 6232 checkpoint started...
2026-05-15 04:33:38.564+0000 INFO  [o.n.k.i.t.l.p.LogPruningImpl] [neo4j/845e4053] No log version pruned. The strategy used was '2 days 2147483648 size'. 
2026-05-15 04:33:39.564+0000 INFO  [o.n.k.i.t.l.c.CheckPointerImpl] [neo4j/845e4053] Checkpoint triggered by "Scheduled checkpoint for every 1 log chunks threshold" @ txId: 6394, append index: 6394 checkpoint started...
2026-05-15 04:33:39.646+0000 INFO  [o.n.k.i.t.l.p.LogPr

## Demo 3 — Crash recovery: committed survives, uncommitted vanishes

Two textbook properties in one demo:

- **Durability.** Auto-commit a `:CrashMarker {status: 'committed'}`. Its WAL record has a COMMIT entry → recovery replays it.
- **Crash atomicity.** Open a tx, write a `:CrashMarker {status: 'uncommitted'}`, and never `commit()`. Its data sits in heap only — there's nothing on disk for recovery to find.

Then `SIGKILL` the container, restart, and query both markers. Committed must be present; uncommitted must be gone.

This is the demo Aura cannot run.

In [4]:
committed_id   = f"{RUN_ID}-COMMITTED"
uncommitted_id = f"{RUN_ID}-UNCOMMITTED"

# 1. Auto-commit marker
with session() as s:
    s.run("""
        CREATE (:CrashMarker:DemoNode {
            marker_id: $m, status: 'committed', written_at: datetime()
        })
    """, m=committed_id).consume()
print(f"COMMITTED   : {committed_id}  ← should survive")

# 2. Open a tx, write, but DO NOT commit 
limbo_session = driver.session(database=DB)
limbo_tx = limbo_session.begin_transaction()
limbo_tx.run("""
    CREATE (:CrashMarker:DemoNode {
        marker_id: $m, status: 'uncommitted', written_at: datetime()
    })
""", m=uncommitted_id)
print(f"UNCOMMITTED : {uncommitted_id}  ← should NOT survive (tx still open)")

# 3. kill neo4j process 
subprocess.run(["docker", "kill", "-s", "SIGKILL", CONTAINER],
               check=True, capture_output=True)
print("SIGKILL sent")
for closer in (limbo_tx, limbo_session, driver):
    try: closer.close()
    except Exception: pass

# 4. Restart, reconnect.
subprocess.run(["docker", "start", CONTAINER], check=True, capture_output=True)
print("Container restarted; waiting for Bolt...", end="", flush=True)
t0 = time.perf_counter()
driver = None
for _ in range(60):
    try:
        driver = neo4j.GraphDatabase.driver(URI, auth=(USER, PWD), connection_timeout=2)
        driver.verify_connectivity()
        break
    except Exception:
        if driver: driver.close()
        driver = None
        print(".", end="", flush=True)
        time.sleep(0.5)
else:
    raise RuntimeError("Bolt didn't come back in 60 polls")
print(f" up in {time.perf_counter() - t0:.1f}s\n")

show_log("debug.log", n=30, grep="recover|replay")

# 5. Verify
with session() as s:
    found_c = s.run("MATCH (m:CrashMarker {marker_id: $m}) RETURN m.marker_id AS id",
                    m=committed_id).single()
    found_u = s.run("MATCH (m:CrashMarker {marker_id: $m}) RETURN m.marker_id AS id",
                    m=uncommitted_id).single()
print(f"\nCommitted marker present?   {found_c is not None}   ({found_c})")
print(f"Uncommitted marker present? {found_u is not None}   ({found_u})")

COMMITTED   : a20135ef-COMMITTED  ← should survive
UNCOMMITTED : a20135ef-UNCOMMITTED  ← should NOT survive (tx still open)


[#CDFC]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('localhost', 7687)) (ResolvedIPv6Address(('::1', 7687, 0, 0))): ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None)


SIGKILL sent
Container restarted; waiting for Bolt..................... up in 9.4s

--- /logs/debug.log  (recover|replay) ---
2026-05-15 04:27:15.801+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  20% completed
2026-05-15 04:27:15.801+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  30% completed
2026-05-15 04:27:15.801+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  40% completed
2026-05-15 04:27:15.801+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  50% completed
2026-05-15 04:27:16.263+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  60% completed
2026-05-15 04:27:16.263+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  70% completed
2026-05-15 04:27:16.264+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  80% completed
2026-05-15 04:27:16.264+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053]  90% completed
2026-05-15 04:27:16.264+0000 INFO  [o.n.k.r.Recovery] [neo4j/845e4053] 100% completed
2026-05-15 04:27:16.458+0000 INFO  [o.n.k.d.Database] [neo4j/845e4053] Recovery in 'full' mode compl

## Demo 4 — Non-repeatable read

Neo4j default isolation = READ COMMITTED: no dirty reads, but the same value read twice in one tx can change if another tx commits between the reads.

```
READER tx:  read#1 -------------- read#2 -- commit
WRITER tx:           SET -- commit
```

Threads sync via `Event`s, so the timing — and therefore `r1 != r2` — is deterministic.

In [5]:
with session() as s:
    s.run("MATCH (p:PATIENT {patient_id: $pid}) SET p.demo_note = 'initial'",
          pid=PID).consume()

phase1, phase2 = Event(), Event()
reads = {}


def reader():
    with session() as s, s.begin_transaction() as tx:
        reads["r1"] = tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                             "RETURN p.demo_note AS n", pid=PID).single()["n"]
        print(f"reader read#1 = {reads['r1']!r}")
        phase1.set()
        phase2.wait(timeout=10)
        reads["r2"] = tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                             "RETURN p.demo_note AS n", pid=PID).single()["n"]
        print(f"reader read#2 = {reads['r2']!r}   (same tx, after writer commit)")


def writer():
    phase1.wait()
    with session() as s:
        s.run("MATCH (p:PATIENT {patient_id: $pid}) "
              "SET p.demo_note = 'mutated'", pid=PID).consume()
    print("writer committed")
    phase2.set()


a, b = Thread(target=reader), Thread(target=writer)
a.start(); b.start(); a.join(); b.join()

assert reads["r1"] != reads["r2"], "non-repeatable read did not occur"
print(f"\n✓ non-repeatable read: r1={reads['r1']!r}  r2={reads['r2']!r}\n")

show_log("query.log", grep="demo_note")

reader read#1 = 'initial'
writer committed
reader read#2 = 'mutated'   (same tx, after writer commit)

✓ non-repeatable read: r1='initial'  r2='mutated'

--- /logs/query.log  (demo_note) ---
(no matches)


## Demo 5 — Lost update: naive RMW vs atomic SET

Read-modify-write across two statements takes no lock on the read. Two threads can both read `v`, both write `v+1`, and one increment vanishes silently — no error.

The naive thread `sleep(1)`s between read and write. With network roundtrips well under a second, both threads always finish reading before either writes — the race fires deterministically:

| Round  | Pattern                            | Expected | Actual |
|--------|------------------------------------|----------|--------|
| Naive  | `v = SELECT; sleep 1s; SET = v+1`  | 20       | **10** |
| Atomic | `SET p.x = p.x + 1`                | 20       | **20** |

Retries can't help — only one-statement Cypher prevents the anomaly.

Runtime: naive round ≈ 10 s (`N=10` × 1 s sleep). Atomic round is instant.

In [6]:
N = 10  # naive thread sleeps 1s per iteration -- keep small so the cell is fast


def naive_thread():
    with session() as s:
        for _ in range(N):
            with s.begin_transaction() as tx:
                v = tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                           "RETURN coalesce(p.demo_visit_count, 0) AS v",
                           pid=PID).single()["v"]
                # time.sleep(0.1)  # both threads' reads land before either writes
                tx.run("MATCH (p:PATIENT {patient_id: $pid}) "
                       "SET p.demo_visit_count = $v", pid=PID, v=v + 1)


def atomic_thread():
    with session() as s:
        for _ in range(N):
            s.run("MATCH (p:PATIENT {patient_id: $pid}) "
                  "SET p.demo_visit_count = coalesce(p.demo_visit_count, 0) + 1",
                  pid=PID).consume()


def reset_count():
    with session() as s:
        s.run("MATCH (p:PATIENT {patient_id: $pid}) SET p.demo_visit_count = 0",
              pid=PID).consume()


def get_count():
    with session() as s:
        return s.run("MATCH (p:PATIENT {patient_id: $pid}) "
                     "RETURN p.demo_visit_count AS v", pid=PID).single()["v"]


def run_pair(target):
    ts = [Thread(target=target) for _ in range(2)]
    for t in ts: t.start()
    for t in ts: t.join()


reset_count()
print(f"NAIVE (1s sleep between read and write), 2 threads x {N}:")
run_pair(naive_thread)
nv = get_count()
print(f"  -> {nv}   (expected {N}, lost {2*N - nv})\n")
assert nv == N

reset_count()
print(f"ATOMIC SET, 2 threads x {N}:")
run_pair(atomic_thread)
av = get_count()
print(f"  -> {av}   (expected {2*N})\n")
assert av == 2 * N

show_log("query.log", grep="demo_visit_count")
cleanup()
print("\nDemo state cleaned.")

NAIVE (1s sleep between read and write), 2 threads x 10:
  -> 10   (expected 10, lost 10)

ATOMIC SET, 2 threads x 10:
  -> 20   (expected 20)

--- /logs/query.log  (demo_visit_count) ---
(no matches)

Demo state cleaned.
